# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
print("Token added successfully:", hf_token is not None)
file_c = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token)
file_f = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token)
df_c = pd.read_parquet(file_c)
df_f = pd.read_parquet(file_f)
print(df_c.shape, df_f.shape)
df_c.head()



Token added successfully: True
(519606, 26) (9841378, 30)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [12]:
page_stats=df_f.groupby('content_hash_id').agg(
    total_clicks=('gsc_clicks','sum'),
    total_impressions=('gsc_impressions', 'sum'),
    avg_position=('gsc_sum_position', 'mean')
).reset_index()
page_stats

,content_hash_id,total_clicks,total_impressions,avg_position
0,content_000005d4ced12088,0,86,199.967742
1,content_00001e488b74b799,0,0,0.000000
2,content_00007bd2985b77c3,0,47,8.032258
3,content_00008950670cb6b5,0,0,0.000000
4,content_0000a348850eb1fc,0,0,0.000000
...,...,...,...,...
331432,content_ffffbc148e416f89,0,1,3.129032
331433,content_ffffc282ab2cbe62,0,0,0.000000
331434,content_ffffc58385523096,37,2482,293.645161
331435,content_ffffe701567e982c,0,0,0.000000


In [13]:
df_f['report_date']=pd.to_datetime(df_f['report_date'])
max_date=df_f['report_date'].max()
min_date=df_f['report_date'].min()
print(f"The max date is {max_date} & the min date is {min_date}")

The max date is 2026-03-31 00:00:00 & the min date is 2026-03-01 00:00:00


In [14]:
df_c['content_updated_date'] = pd.to_datetime(df_c['content_updated_date'])
kept_rows = df_c[df_c['content_updated_date'] <= max_date].copy()
kept_rows['days_since_update'] = (max_date - kept_rows['content_updated_date']).dt.days
features = ['content_hash_id', 'content_updated_date', 'word_count', 'content_type', 'is_deleted', 'days_since_update']
kept_rows[features]
kept_rows=kept_rows[kept_rows['is_deleted']==False]
kept_rows[features]
len(kept_rows)

42821

In [15]:
df=kept_rows[features].merge(page_stats,on='content_hash_id',how='inner')
df.head()

,content_hash_id,content_updated_date,word_count,content_type,is_deleted,days_since_update,total_clicks,total_impressions,avg_position
0,content_04c67f3541177192,2026-02-25,3168.0,keyword article,False,34,2,331,153.516129
1,content_05acc92c165f4386,2026-02-25,4135.0,keyword article,False,34,0,33,9.967742
2,content_0b33d8960857ad90,2026-02-25,3232.0,keyword article,False,34,0,0,0.000000
3,content_0f30e04e709c7b5d,2026-02-25,3211.0,keyword article,False,34,0,145,38.000000
4,content_167472cd0802a8f3,2026-02-25,3149.0,keyword article,False,34,0,232,89.516129


In [16]:
df['is_page_one'] = (df['avg_position'] <= 10).astype(int)
df['is_stale'] = (df['days_since_update'] >= 180).astype(int)
df['opportunity_flags'] = df['is_page_one'] + df['is_stale']
df['opportunity_flags'].value_counts().sort_index()

opportunity_flags
0    21199
1    12624
2     3558
Name: count, dtype: int64

In [17]:
dup=df['content_hash_id'].duplicated().sum()
print(f"There are {dup} duplicates in dataframe")
shape=df.shape
print(f"There are {shape[0]} rows and {shape[1]} columns in dataset")
print(df.notnull().sum())

There are 0 duplicates in dataframe
There are 37381 rows and 12 columns in dataset
content_hash_id         37381
content_updated_date    37381
word_count              24799
content_type            37381
is_deleted              37381
days_since_update       37381
total_clicks            37381
total_impressions       37381
avg_position            37381
is_page_one             37381
is_stale                37381
opportunity_flags       37381
dtype: int64


In [18]:
df['word_count']=df['word_count'].fillna(df['word_count'].median())
df.isnull().sum()

content_hash_id         0
content_updated_date    0
word_count              0
content_type            0
is_deleted              0
days_since_update       0
total_clicks            0
total_impressions       0
avg_position            0
is_page_one             0
is_stale                0
opportunity_flags       0
dtype: int64

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
bins=[0,90,180,df['days_since_update'].max()]
label=['Fresh','Aging','Stale']
df['staleness_bucket']=pd.cut(df['days_since_update'],bins=bins,labels=label)
df.head()

,content_hash_id,content_updated_date,word_count,content_type,is_deleted,days_since_update,total_clicks,total_impressions,avg_position,is_page_one,is_stale,opportunity_flags,staleness_bucket
0,content_04c67f3541177192,2026-02-25,3168.0,keyword article,False,34,2,331,153.516129,0,0,0,Fresh
1,content_05acc92c165f4386,2026-02-25,4135.0,keyword article,False,34,0,33,9.967742,1,0,1,Fresh
2,content_0b33d8960857ad90,2026-02-25,3232.0,keyword article,False,34,0,0,0.000000,1,0,1,Fresh
3,content_0f30e04e709c7b5d,2026-02-25,3211.0,keyword article,False,34,0,145,38.000000,0,0,0,Fresh
4,content_167472cd0802a8f3,2026-02-25,3149.0,keyword article,False,34,0,232,89.516129,0,0,0,Fresh


In [30]:
df['staleness_bucket'].value_counts()

staleness_bucket
Fresh    30198
Stale     3600
Aging     3583
Name: count, dtype: int64

In [33]:
df.groupby('staleness_bucket')['total_clicks'].agg(['mean','count']).reset_index()

,staleness_bucket,mean,count
0,Fresh,2.541393,30198
1,Aging,0.362545,3583
2,Stale,0.007222,3600


**Signal 1:** Staleness (days_since_update) vs. total_clicks: Verdict = CONFIRMED. Mean clicks drop sharply as staleness increases. Fresh pages average 2.54 clicks, Aging pages 0.36, Stale pages 0.007 (n = 30,198 / 3,583 / 3,600). This supports using staleness as a real signal in the rule.

In [41]:
bins=[0,0.001,10,30,df['avg_position'].max()]
labels=['No data','Top 10','11-30','30+']
df['avg_position_bucket']=pd.cut(df['avg_position'], bins=bins,labels=labels, include_lowest=True)
df.head()

,content_hash_id,content_updated_date,word_count,content_type,is_deleted,days_since_update,total_clicks,total_impressions,avg_position,is_page_one,is_stale,opportunity_flags,staleness_bucket,avg_position_bucket
0,content_04c67f3541177192,2026-02-25,3168.0,keyword article,False,34,2,331,153.516129,0,0,0,Fresh,30+
1,content_05acc92c165f4386,2026-02-25,4135.0,keyword article,False,34,0,33,9.967742,1,0,1,Fresh,Top 10
2,content_0b33d8960857ad90,2026-02-25,3232.0,keyword article,False,34,0,0,0.000000,1,0,1,Fresh,No data
3,content_0f30e04e709c7b5d,2026-02-25,3211.0,keyword article,False,34,0,145,38.000000,0,0,0,Fresh,30+
4,content_167472cd0802a8f3,2026-02-25,3149.0,keyword article,False,34,0,232,89.516129,0,0,0,Fresh,30+


In [43]:
df.groupby('avg_position_bucket')['total_clicks'].agg(['mean','count']).reset_index()

,avg_position_bucket,mean,count
0,No data,0.000516,9697
1,Top 10,0.053236,6443
2,11-30,0.228053,3030
3,30+,4.229916,18211


In [44]:
df[df['avg_position_bucket'] == '30+'][['total_impressions', 'total_clicks', 'avg_position']].head(10)

,total_impressions,total_clicks,avg_position
0,331,2,153.516129
3,145,0,38.000000
4,232,0,89.516129
7,464,0,141.258065
9,36,0,30.064516
11,774,5,318.258065
15,311,0,85.677419
21,248,0,173.806452
22,3547,21,763.838710
27,2092,1,3586.645161


**Signal 2:** Position (avg_position) vs. total_clicks: Verdict = MIXED / UNRELIABLE. The raw numbers actually run backwards. Top 10 pages average 0.05 clicks while 30+ pages average 4.23 clicks (n = 6,443 vs 18,211)the opposite of what CTR-vs-position theory predicts. Investigating individual rows shows why: several "30+" pages have avg_position values in the hundreds or thousands (e.g., 3,586), which isn't a real search ranking it's an artifact of how avg_position was built (mean of gsc_sum_position, a daily sum across all queries, not a properly impression-weighted average a limitation already flagged in the ML-04 data contract). This signal can't be trusted as is for the rule it would need a corrected, impression-weighted position calculation before being used confidently.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def get_action(bucket):
    if bucket == 'Fresh':
        return 'no action needed'
    elif bucket == 'Aging':
        return 'monitor'
    else:
        return 'refresh'
df['action']=df['staleness_bucket'].apply(get_action)

df[['staleness_bucket','action']].tail(10)


,staleness_bucket,action
37371,Fresh,no action needed
37372,Fresh,no action needed
37373,Fresh,no action needed
37374,Fresh,no action needed
37375,Fresh,no action needed
37376,Fresh,no action needed
37377,Fresh,no action needed
37378,Fresh,no action needed
37379,Fresh,no action needed
37380,Fresh,no action needed


In [64]:
def get_reason_code(bucket):
    if bucket=='Fresh':
        return 'fresh, no action needed'
    elif bucket=='Stale':
        return 'Stale content'
    else:
        return 'the page is old'
df['reason_code']=df['staleness_bucket'].apply(get_reason_code)
df[['staleness_bucket','action','reason_code']].head()



,staleness_bucket,action,reason_code
0,Fresh,no action needed,"fresh, no action needed"
1,Fresh,no action needed,"fresh, no action needed"
2,Fresh,no action needed,"fresh, no action needed"
3,Fresh,no action needed,"fresh, no action needed"
4,Fresh,no action needed,"fresh, no action needed"


In [73]:
df['score']=df['days_since_update']
df[['staleness_bucket','score','action','reason_code']].head()
sorted=df.sort_values(by='score',ascending=False)
sorted.to_csv('../output/baseline_action_score.csv',index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.